# Regresión Lineal - Accidentes de tránsito en Bogotá

## Notebook 1: Validación y limpieza de datos

Documenta la exploración de los datos crudos, la evidencia que motivó cada decisión de limpieza, y deja el dataset limpio guardado para el siguiente notebook (`02_agregacion_features.ipynb`).

In [1]:
# Permite importar los módulos de src/helpers desde este notebook
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent / "src"))

from helpers.generate_dataframe import dataframe
from helpers.clean_dataframe import clean_dataframe

## 1. Carga de datos crudos

Cargamos el CSV histórico de siniestros tal como viene, sin transformar.

In [2]:
df = dataframe()
df.shape

(199146, 16)

In [3]:
df.head()

,ï»¿X,Y,OBJECTID,FORMULARIO,CODIGO_ACCIDENTE,FECHA_OCURRENCIA_ACC,ANO_OCURRENCIA_ACC,DIRECCION,GRAVEDAD,CLASE_ACC,LOCALIDAD,FECHA_HORA_ACC,LATITUD,LONGITUD,CIV,PK_CALZADA
0,-74.090.923.953,469.380.685.384.595,1,A000640275,4484660,2017/06/12 00:00:00+00,2017,AV AVENIDA BOYACA-CL 79 02,SOLO DANOS,CHOQUE,ENGATIVA,2017/06/12 05:30:00+00,469.380.685,-7.409.092.395,10006772.0,221236.0
1,-74.121,460.299.999.984.892,2,A001233353,10533499,2020/11/19 00:00:00+00,2020,CL 26 S- KR 50 02,CON HERIDOS,OTRO,PUENTE ARANDA,2020/11/19 02:05:00+00,4.603,-74.121,16004560.0,NaN
2,-74.042,468.199.999.984.635,4,A001232786,10533629,2020/11/10 00:00:00+00,2020,KR 9 - CL 100 02,SOLO DANOS,CHOQUE,USAQUEN,2020/11/10 13:30:00+00,4.682,-74.042,30001107.0,NaN
3,-741.669.374.849.999,45.871.866.868.494,7,A000200705,4412699,2015/05/11 00:00:00+00,2015,CL 63A-KR 72 S 02,SOLO DANOS,CHOQUE,CIUDAD BOLIVAR,2015/05/11 10:50:00+00,458.718.669,-7.416.693.749,19001483.0,136166.0
4,-740.929.008.909.999,460.764.758.184.872,8,A000402862,4447845,2016/06/08 00:00:00+00,2016,KR 27-CL 9 14,SOLO DANOS,CHOQUE,LOS MARTIRES,2016/06/08 21:30:00+00,460.764.758,-7.409.290.089,14000548.0,239719.0


**Nota sobre calidad de datos:** las columnas de coordenadas (`ï»¿X`, `Y`, `LATITUD`, `LONGITUD`) muestran valores mal formateados (p. ej. `-74.090.923.953`), aparentemente por una exportación en configuración regional que usa `.` como separador de miles. No las corregimos aquí porque el modelo de este proyecto no las usa (trabajamos a nivel de fecha y `LOCALIDAD`), pero quedan como pendiente si en el futuro se quiere hacer análisis geoespacial (mapas de calor de accidentes, clustering espacial, etc.).

## 2. Exploración inicial

Revisamos qué localidades existen y cuántos registros tiene cada una, incluyendo nulos.

In [4]:
df['LOCALIDAD'].unique()

<StringArray>
[          'ENGATIVA',      'PUENTE ARANDA',            'USAQUEN',
     'CIUDAD BOLIVAR',       'LOS MARTIRES',               'SUBA',
           'FONTIBON',               'USME',        'TEUSAQUILLO',
     'BARRIOS UNIDOS', 'RAFAEL URIBE URIBE',          'CHAPINERO',
            'KENNEDY',     'ANTONIO NARINO',               'BOSA',
      'SAN CRISTOBAL',         'CANDELARIA',         'TUNJUELITO',
           'SANTA FE',                  nan,            'SUMAPAZ']
Length: 21, dtype: str

In [5]:
df['LOCALIDAD'].value_counts(dropna=False)

LOCALIDAD
KENNEDY               23661
ENGATIVA              20928
USAQUEN               19292
SUBA                  18973
FONTIBON              16377
PUENTE ARANDA         14143
CHAPINERO             11696
TEUSAQUILLO           10167
BARRIOS UNIDOS        10094
BOSA                   9417
CIUDAD BOLIVAR         8005
LOS MARTIRES           6250
SANTA FE               5451
TUNJUELITO             5351
SAN CRISTOBAL          5335
RAFAEL URIBE URIBE     5333
USME                   4017
ANTONIO NARINO         3721
CANDELARIA              882
NaN                      46
SUMAPAZ                   7
Name: count, dtype: int64

**Hallazgos:**
- Hay 46 registros sin localidad asignada (`NaN`).
- `SUMAPAZ` tiene solo 7 registros, un caso atípico frente al resto de localidades.

Ambos casos se descartan en la limpieza (ver `clean_dataframe`).

## 3. Limpieza de datos

La función `clean_dataframe` (en `src/helpers/clean_dataframe.py`) aplica:
1. Conversión de `FECHA_HORA_ACC` a tipo fecha (`pd.to_datetime`, con `errors='coerce'` para no romper con fechas inválidas).
2. Filtro de registros sin `LOCALIDAD` o con `LOCALIDAD == 'SUMAPAZ'`.

In [5]:
df_clean = clean_dataframe(df)
df_clean.shape

(199093, 16)

## 4. Guardar dataset limpio

Persistimos el resultado en `data/processed/` para que el siguiente notebook (agregación y features) parta de aquí sin tener que repetir la carga y limpieza.

In [6]:
ruta_salida = Path.cwd().parent / "data" / "processed" / "accidentes_limpios.csv"
ruta_salida.parent.mkdir(parents=True, exist_ok=True)

df_clean.to_csv(ruta_salida, index=False)
ruta_salida

PosixPath('/Users/jampier/Desktop/ESP. CIENCIA_DE_DATOS/Machine learning/Regresion_lineal/data/processed/accidentes_limpios.csv')

## Resumen y próximos pasos

- Se cargaron 199.146 registros crudos de accidentes de tránsito en Bogotá (2015-2021).
- Se descartaron 53 registros en la limpieza: 46 sin `LOCALIDAD` asignada y 7 de `SUMAPAZ` (localidad con muy pocos registros frente al resto, un caso atípico).
- El dataset limpio queda en 199.093 registros, con `FECHA_HORA_ACC` ya convertida a tipo fecha.
- Resultado guardado en `data/processed/accidentes_limpios.csv`.

Continúa en `02_agregacion_features.ipynb`: agregación diaria por localidad, relleno de combinaciones fecha-localidad sin accidentes, e ingeniería de features de fecha (día de la semana, mes, año).